** Gerei a chave usando o Google AI Studio e rodei dentro do proprio ambiente**

Rodei o codigo dentro do google colab, sendo assim a chave da API eu passei dentro da parte de "Secrets" que fica na propria extensão do colab se for for rodar local tem que passar a chave no proprio codigo

In [3]:
!pip install google-genai pydantic

In [6]:
import os
from google.colab import userdata

# 3. Carregar o segredo para a variável de ambiente
# LEMBRETE: Sua chave deve estar configurada no painel de Segredos (chave 🔑)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
print("Configuração de bibliotecas e chave API concluída.")

Configuração de bibliotecas e chave API concluída.


In [9]:
class Entidade(BaseModel):
    """Estrutura para uma entidade nomeada."""
    texto: str = Field(description="O texto exato da entidade encontrada.")
    tipo: str = Field(description="A classificação da entidade, como PESSOA, ORGANIZACAO, LOCAL, DATA, MISC, etc.")

class EntidadesEncontradas(BaseModel):
    """Lista de todas as entidades nomeadas no texto."""
    entidades: List[Entidade] = Field(description="Lista de todas as entidades nomeadas identificadas pelo modelo.")

# --- 2. Função Principal de Análise ---

def identificar_entidades_llm(texto_input: str) -> dict:
    """
    Usa o modelo Gemini para identificar e classificar entidades nomeadas em um texto.
    """

    # 2a. Inicializar o Cliente Gemini
    try:
        if not os.getenv("GEMINI_API_KEY"):
             raise ValueError("Variável de ambiente GEMINI_API_KEY não está configurada.")

        client = genai.Client()
    except Exception as e:
        return {"erro": "Falha na inicialização do cliente LLM. Verifique sua chave API."}

    # 2b. O Prompt Guia
    # O prompt é o que mudamos para focar na nova tarefa.
    prompt = f"""
    Analise o texto abaixo e identifique todas as entidades nomeadas.
    Para cada entidade, retorne o texto e o tipo de entidade (por exemplo: PESSOA, ORGANIZACAO, LOCAL, DATA, MISC).
    Retorne apenas entidades nomeadas importantes.

    TEXTO A SER ANALISADO:
    ---
    {texto_input}
    ---
    """

    # 2c. Configuração da Chamada da API
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=EntidadesEncontradas, # Usando o novo Schema
    )

    # 2d. Chamada da API
    print("-> Enviando texto para identificação de Entidades...")
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=config,
        )

        resultado_json = response.text
        return json.loads(resultado_json)

    except Exception as e:
        print(f"Ocorreu um erro durante a chamada da API: {e}")
        return {"erro": str(e)}

# --- 3. Execução Interativa ---

if __name__ == "__main__":

    # Texto de Exemplo para NER
    texto_ner_exemplo = (
        "A Dra. Maria Silva, CEO da TechCorp, viajou para Paris em 15 de janeiro. "
        "A empresa, sediada em São Paulo, anunciou novos projetos."
    )

    print("\n" + "="*70)
    print("✨ EXERCÍCIO 2: IDENTIFICAÇÃO DE ENTIDADES NOMEADAS (NER) ✨")
    print("="*70)

    # 3a. Pede o texto ao usuário
    print("Digite o texto para Identificação de Entidades (ou ENTER para usar o exemplo padrão):")
    texto_input_usuario = input(f"> ")

    texto_final = texto_input_usuario.strip() if texto_input_usuario.strip() else texto_ner_exemplo

    # 4. Obter e Imprimir o Resultado
    resultado_analise = identificar_entidades_llm(texto_final)

    print("\n" + "-" * 70)
    print(f"TEXTO ORIGINAL:\n{texto_final}")
    print("-" * 70)

    if "erro" in resultado_analise:
        print(f"STATUS: FALHA\n{resultado_analise['erro']}")
    else:
        print(f"STATUS: SUCESSO. {len(resultado_analise.get('entidades', []))} Entidades Encontradas:")
        print("-" * 70)

        # Imprime a tabela de resultados
        print("{:<20} {:<20}".format('ENTIDADE', 'TIPO'))
        print("{:<20} {:<20}".format('-' * 18, '-' * 18))
        for entidade in resultado_analise.get('entidades', []):
            print("{:<20} {:<20}".format(entidade.get('texto'), entidade.get('tipo')))

        print("-" * 70)
        print("Estrutura JSON Completa:")
        print(json.dumps(resultado_analise, indent=2, ensure_ascii=False))


✨ EXERCÍCIO 2: IDENTIFICAÇÃO DE ENTIDADES NOMEADAS (NER) ✨
Digite o texto para Identificação de Entidades (ou ENTER para usar o exemplo padrão):
> Geovani foi jogar futebol
-> Enviando texto para identificação de Entidades...

----------------------------------------------------------------------
TEXTO ORIGINAL:
Geovani foi jogar futebol
----------------------------------------------------------------------
STATUS: SUCESSO. 1 Entidades Encontradas:
----------------------------------------------------------------------
ENTIDADE             TIPO                
------------------   ------------------  
Geovani              PESSOA              
----------------------------------------------------------------------
Estrutura JSON Completa:
{
  "entidades": [
    {
      "texto": "Geovani",
      "tipo": "PESSOA"
    }
  ]
}
